In [1]:
import numpy as np
import pandas as pd
import csv
import matplotlib.pyplot as plt

%matplotlib inline
%matplotlib notebook

### 1.Load  csv file.

In [2]:
# load the csv file.
path = 'C:/Users/ZWX/PythonNotebooks/UWBM/Unittest/OpenPaved/'
InputData = pd.read_csv(path + 'input_csv.csv')

In [3]:
date = InputData['date']
P_atm = InputData['P_atm']
Ref_grass = InputData['Ref.grass']
E_pot_OW = InputData['E_pot_OW']

### 2. OpenPaved ###

In [4]:
# iters = Total timestep.
iters = np.shape(date)[0]

#### 2.1 General test (Default settings)

In [5]:
# Given conditions.

# tot_area --- total area of study domain
tot_area = 10000

# op_frac --- openpaved percentage
op_frac = 5184*0.3048**2/10000
tot_op_area = tot_area * op_frac

# op_meas_area --- area of measure on open paved
# op_nomeas_area --- area of open paved (without a measure)
op_meas_area = 0  
op_no_meas_area = tot_op_area - op_meas_area


# inflowfac --- inflow factor
# measure_inflow_area --- runoff inflow area to measure, inflow area >= measure area, predefined as 0.
measure_inflow_area = 0 
inflowfac_op = (measure_inflow_area - op_meas_area) / op_no_meas_area

In [6]:
print(inflowfac_op)

0.0


In [7]:
class OpenPaved:
    def __init__(self, init_intstor_op, intstorcap_op = 1.6, stormfrac_op = 1.0, discfrac_op = 0.0, infilcap_op = 1):
        
        # state
        self.init_intstor_op = init_intstor_op
        
        # parameter
        # intstorcap_op --- predefined storage capacity on open paved
        # stormfrac_op --- part of urban area with storm water drainage system
        # discfrac_op --- part of open paved area that is disconnected
        # self.mxdfrac--- part of urban area with mixed sewer system
        # infilcap_op --- predefined infiltration capacity [mm/d] on open paved area
        
        self.intstorcap = intstorcap_op
        self.stormfrac = stormfrac_op
        self.mxdfrac = 1 - self.stormfrac
        self.discfrac = discfrac_op
        self.infilcap = infilcap_op

    def __repr__(self):
        return 'Current P is ' + str(p_atm) + 'Current E is ' + str(e_pot_ow) + '.These are input information.'
    
    def sol(self, p_atm, e_pot_ow):
        
        # parameters
        # int_op --- Interception on open paved after rainfall during current time step [mm]
        # e_atm_op --- Evaporation from interception storage on open paved during current time step [mm]
        # intstor_op --- Remaining interception storage on open paved at the end of the current time step [mm]
        # p_op_gw --- Percolation of interception storage on open paved to groundwater during the current time step [mm].
        # r_op_meas --- Runoff from open paved to an area with a drainage measure (not necessarily on the open paved area itself) [mm].
        # r_op_swds --- Runoff from open paved to the storm water drainage system [mm]
        # r_op_mss --- Runoff from open paved to the mixed sewer system [mm]
        # r_op_up --- Runoff from open paved to unpaved area [mm].
        
        if op_no_meas_area == 0:
            int_op = e_atm_op = intstor_op = p_op_gw = r_op_meas = r_op_swds = r_op_mss = r_op_up = 0
            
        else:
            int_op = min(self.intstorcap, max(0, p_atm + self.init_intstor_op))

            e_atm_op = min(e_pot_ow, int_op)
            
            intstor_op = int_op - e_atm_op
            
            p_op_gw = max(0, min(p_atm - (self.intstorcap - self.init_intstor_op), self.infilcap * delta_t)) # infiltration capacity (mm/d) * time step size (hr to d)
            
            r_op_meas = inflowfac_op * max(0, p_atm - e_atm_op - (intstor_op - self.init_intstor_op) - p_op_gw)
            
            r_op_swds = self.stormfrac * (1 - self.discfrac) * max(0, p_atm - e_atm_op - (intstor_op - self.init_intstor_op) - p_op_gw - r_op_meas)
            
            r_op_mss = self.mxdfrac * (1 - self.discfrac) * max(0, p_atm - e_atm_op - (intstor_op - self.init_intstor_op) - p_op_gw - r_op_meas)
            
            r_op_up = self.discfrac * max(0, p_atm - e_atm_op - (intstor_op - self.init_intstor_op) - p_op_gw - r_op_meas)
            
            # update state
            self.init_intstor_op = intstor_op
        
        return int_op, e_atm_op, intstor_op, p_op_gw, r_op_meas, r_op_swds, r_op_mss, r_op_up

In [8]:
# Database for all unknown. ([0] is used to fill the vacancy at time level t = 0.) 

delta_t = 1 / 24

E_atm = [0]
Intcp = [0]
IntStor = [0]
P_gw = [0]
R_meas = [0]
R_swds = [0]
R_mss = [0]
R_up = [0]

m = OpenPaved(init_intstor_op = 0, intstorcap_op = 1.6, stormfrac_op = 1.0, discfrac_op = 0.0, infilcap_op = 1.0)

t = 1

while t <= iters-1:
    
    sol = m.sol(P_atm[t], E_pot_OW[t])
    
    Intcp.append(sol[0])
    E_atm.append(sol[1])
    IntStor.append(sol[2])
    P_gw.append(sol[3])
    R_meas.append(sol[4])
    R_swds.append(sol[5])
    R_mss.append(sol[6])
    R_up.append(sol[7])
    
    t += 1
    
filename = 'OP_General_test_pysol.csv'
np.savetxt('pysol/' + filename, np.c_[Intcp, E_atm, IntStor, P_gw, R_meas, R_swds, R_mss, R_up], fmt = "%.8f", delimiter=',', header = 'Intcp, E_atm, IntStor, P_gw, R_meas, R_swds, R_mss, R_up')
df = pd.read_csv('pysol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('pysol/' + filename)

##### 2.1.1 Validation

In [9]:
# read python file
data_py = pd.read_csv('pysol/' + filename)

# read excel file
data_ex = pd.read_csv('exsol/OP_General_test_exsol.csv')

In [10]:
# Examine (go through all the data)
database = []
# from col 0 to last col (col 6)
for c in range(8):
    for r in range(1,43825): # from row 1 to the last row (row 43824)
        a = data_ex[list(data_ex)[c]][r] - data_py[list(data_py)[c+2]][r]
        database.append(a)
print(max(database))
print(min(database))

5.00000019165725e-09
-5.000000413701855e-09


In [11]:
# Examine (go through column by colum)
A = np.zeros((43825, 8))
for c in range(8):
    for r in range(1,43825): # do not include the initial row.
        A[r,c] = data_ex[list(data_ex)[c]][r] - data_py[list(data_py)[c+2]][r]
for c in range(8):
    print('col ' + str(c), 'max', max(A[:,c]), 'min', min(A[:,c]))

col 0 max 5.00000019165725e-09 min -5.00000019165725e-09
col 1 max 5.000000025123796e-09 min -5.000000025123796e-09
col 2 max 5.00000019165725e-09 min -5.00000019165725e-09
col 3 max 5.000000000837668e-09 min -4.0000000013640236e-09
col 4 max 0.0 min 0.0
col 5 max 4.000000330961484e-09 min -5.000000413701855e-09
col 6 max 0.0 min 0.0
col 7 max 0.0 min 0.0


#### 2.2 Extended test (Different coefficient sets)

Set 1: intstorcap_cp = 0

Set 2: intstorcap_cp = 1600

Set 3: stormfrac = 0.0

Set 4: stormfrac = 0.37

Set 5: discfrac = 1.0

Set 6: discfrac = 0.37

Set 7: infilcap_op = 0

Set 8: infilcap_op = 100

Set 9: cp_no_meas_area = 0

In [12]:
def validatefun(a, b, c, d, e): 
    
    delta_t = 1 / 24

    E_atm = [0]
    Intcp = [0]
    IntStor = [0]
    P_gw = [0]
    R_meas = [0]
    R_swds = [0]
    R_mss = [0]
    R_up = [0]

    m = OpenPaved(init_intstor_op = 0, intstorcap_op = a, stormfrac_op = b, discfrac_op = c, infilcap_op = d)

    t = 1

    while t <= iters-1:
    
        sol = m.sol(P_atm[t], E_pot_OW[t])
    
        Intcp.append(sol[0])
        E_atm.append(sol[1])
        IntStor.append(sol[2])
        P_gw.append(sol[3])
        R_meas.append(sol[4])
        R_swds.append(sol[5])
        R_mss.append(sol[6])
        R_up.append(sol[7])
    
        t += 1
    
    filename = 'OP_extended_test_pysol_set'+str(e)+'.csv'
    np.savetxt('pysol/' + filename, np.c_[Intcp, E_atm, IntStor, P_gw, R_meas, R_swds, R_mss, R_up], fmt = "%.8f", delimiter=',', header = 'Intcp, E_atm, IntStor, P_gw, R_meas, R_swds, R_mss, R_up')
    df = pd.read_csv('pysol/' + filename)
    df.insert(0, 'Date', date)
    df.to_csv('pysol/' + filename)

    data_py = pd.read_csv('pysol/' + filename)
    data_ex = pd.read_csv('exsol/OP_extended_test_exsol_set'+str(e)+'.csv')
    A = np.zeros((43825, 8))
    for c in range(8):
        for r in range(1,43825):
            A[r,c] = data_ex[list(data_ex)[c]][r] - data_py[list(data_py)[c+2]][r]
    for c in range(8):
        print('COL ' + str(c), 'max', max(A[:,c]), 'min', min(A[:,c]))
        #print(np.where(max(A[:,c]) != 0 and A[:,c] == max(A[:,c])), np.where(min(A[:,c]) != 0 and A[:,c] == min(A[:,c])))
    return 

__intstorcap_op = a, stormfrac_op = b, discfrac_op = c, infilcap_op = d, set number = e__

##### 2.2.1 Set 1: intstorcap_op = 0

In [13]:
validatefun(0, 1, 0, 1, 1)

COL 0 max 0.0 min 0.0
COL 1 max 0.0 min -3.0007100000000002e-15
COL 2 max 3.0007100000000002e-15 min 0.0
COL 3 max 3.0007100000000002e-15 min -2.9999999984209325e-09
COL 4 max 0.0 min 0.0
COL 5 max 3.000000248221113e-09 min 0.0
COL 6 max 0.0 min 0.0
COL 7 max 0.0 min 0.0


##### 2.2.2 Set 2: intstorcap_op = 1600

In [14]:
validatefun(1600, 1, 0, 1, 2)

COL 0 max 5.000001692678779e-07 min -5.000001692678779e-07
COL 1 max 5.000000025123796e-09 min -5.000000025123796e-09
COL 2 max 5.000001692678779e-07 min -5.000001692678779e-07
COL 3 max 3.999999997894577e-09 min -2.9999999984209325e-09
COL 4 max 0.0 min 0.0
COL 5 max 4.000000330961484e-09 min -5.000000413701855e-09
COL 6 max 0.0 min 0.0
COL 7 max 0.0 min 0.0


##### 2.2.3 Set 3: stormfrac = 0.0

In [15]:
validatefun(1.6, 0, 0, 1, 3)

COL 0 max 5.00000019165725e-09 min -5.00000019165725e-09
COL 1 max 5.000000025123796e-09 min -5.000000025123796e-09
COL 2 max 5.00000019165725e-09 min -5.00000019165725e-09
COL 3 max 5.000000000837668e-09 min -4.0000000013640236e-09
COL 4 max 0.0 min 0.0
COL 5 max 0.0 min 0.0
COL 6 max 4.000000330961484e-09 min -5.000000413701855e-09
COL 7 max 0.0 min 0.0


##### 2.2.4 Set 4: stormfrac = 0.37

In [16]:
validatefun(1.6, 0.37, 0, 1, 4)

COL 0 max 5.00000019165725e-09 min -5.00000019165725e-09
COL 1 max 5.000000025123796e-09 min -5.000000025123796e-09
COL 2 max 5.00000019165725e-09 min -5.00000019165725e-09
COL 3 max 5.000000000837668e-09 min -4.0000000013640236e-09
COL 4 max 0.0 min 0.0
COL 5 max 5.000000413701855e-09 min -5.000000413701855e-09
COL 6 max 5.000000413701855e-09 min -5.000000413701855e-09
COL 7 max 0.0 min 0.0


##### 2.2.5 Set 5: discfrac = 1.0

In [17]:
validatefun(1.6, 1, 1, 1, 5)

COL 0 max 5.00000019165725e-09 min -5.00000019165725e-09
COL 1 max 5.000000025123796e-09 min -5.000000025123796e-09
COL 2 max 5.00000019165725e-09 min -5.00000019165725e-09
COL 3 max 5.000000000837668e-09 min -4.0000000013640236e-09
COL 4 max 0.0 min 0.0
COL 5 max 0.0 min 0.0
COL 6 max 0.0 min 0.0
COL 7 max 4.000000330961484e-09 min -5.000000413701855e-09


##### 2.2.6 Set 6: discfrac = 0.37

In [18]:
validatefun(1.6, 1, 0.37, 1, 6)

COL 0 max 5.00000019165725e-09 min -5.00000019165725e-09
COL 1 max 5.000000025123796e-09 min -5.000000025123796e-09
COL 2 max 5.00000019165725e-09 min -5.00000019165725e-09
COL 3 max 5.000000000837668e-09 min -4.0000000013640236e-09
COL 4 max 0.0 min 0.0
COL 5 max 5.000000413701855e-09 min -5.000000413701855e-09
COL 6 max 0.0 min 0.0
COL 7 max 5.000000413701855e-09 min -5.000000413701855e-09


##### 2.2.7 Set 7: infilcap_op = 0

In [19]:
validatefun(1.6, 1, 0, 0, 7)

COL 0 max 5.00000019165725e-09 min -5.00000019165725e-09
COL 1 max 5.000000025123796e-09 min -5.000000025123796e-09
COL 2 max 5.00000019165725e-09 min -5.00000019165725e-09
COL 3 max 0.0 min 0.0
COL 4 max 0.0 min 0.0
COL 5 max 1.000000082740371e-08 min -5.000000413701855e-09
COL 6 max 0.0 min 0.0
COL 7 max 0.0 min 0.0


##### 2.2.8 Set 8: infilcap_op = 100

In [20]:
validatefun(1.6, 1, 0, 100, 8)

COL 0 max 5.00000019165725e-09 min -5.00000019165725e-09
COL 1 max 5.000000025123796e-09 min -5.000000025123796e-09
COL 2 max 5.00000019165725e-09 min -5.00000019165725e-09
COL 3 max 5.00000019165725e-09 min -5.000000413701855e-09
COL 4 max 0.0 min 0.0
COL 5 max 4.000000330961484e-09 min -5.000000413701855e-09
COL 6 max 0.0 min 0.0
COL 7 max 0.0 min 0.0


##### 2.2.9 Set 9: cp_no_meas_area = 0

In [22]:
op_no_meas_area = 0
validatefun(1.6, 1, 0, 1, 9)

COL 0 max 0.0 min 0.0
COL 1 max 0.0 min 0.0
COL 2 max 0.0 min 0.0
COL 3 max 0.0 min 0.0
COL 4 max 0.0 min 0.0
COL 5 max 0.0 min 0.0
COL 6 max 0.0 min 0.0
COL 7 max 0.0 min 0.0


In [ ]:
print('The module has been validated in both general test and extended tests')